In [ ]:
# ============================================
# HOTEL BOOKINGS - DATA CLEANING & PREPARATION
# ============================================

# --------------------------------------------
# 1. Import Libraries
# --------------------------------------------

import pandas as pd
import numpy as np

# Optional display settings
pd.set_option('display.max_columns', None)

# --------------------------------------------
# 2. Load Dataset
# --------------------------------------------

# Load dataset
df = pd.read_csv('data/hotel_bookings.csv')

# Preview dataset
print(df.head())

# --------------------------------------------
# 3. Basic Data Exploration
# --------------------------------------------

# Dataset shape
print("Dataset Shape:", df.shape)

# Columns
print("\nColumns:\n")
print(df.columns)

# Data types
print("\nData Types:\n")
print(df.dtypes)

# Missing values
print("\nMissing Values:\n")
print(df.isnull().sum())

# Duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# --------------------------------------------
# 4. Data Cleaning
# --------------------------------------------

# ---------- Remove Duplicates ----------

df = df.drop_duplicates()

print("\nShape After Removing Duplicates:", df.shape)

# ---------- Handle Missing Values ----------

# Children
df['children'] = df['children'].fillna(0)

# Country
df['country'] = df['country'].fillna('Unknown')

# Agent & Company
# Keep missing values because not all bookings use agents/companies

# ---------- Fix Data Types ----------

# Convert reservation_status_date to datetime
df['reservation_status_date'] = pd.to_datetime(
    df['reservation_status_date'],
    errors='coerce'
)

# Convert is_canceled to integer
df['is_canceled'] = df['is_canceled'].astype(int)

# Convert children to integer
df['children'] = df['children'].astype(int)

# ---------- Standardize Text Columns ----------

text_columns = [
    'hotel',
    'meal',
    'country',
    'market_segment',
    'distribution_channel',
    'deposit_type',
    'customer_type',
    'reservation_status'
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# --------------------------------------------
# 5. Feature Engineering
# --------------------------------------------

# ---------- Total Guests ----------

df['total_guests'] = (
    df['adults'] +
    df['children'] +
    df['babies']
)

# ---------- Total Nights ----------

df['total_nights'] = (
    df['stays_in_weekend_nights'] +
    df['stays_in_week_nights']
)

# ---------- Estimated Revenue ----------

df['estimated_revenue'] = (
    df['adr'] * df['total_nights']
)

# ---------- Has Agent ----------

df['has_agent'] = df['agent'].notnull().astype(int)

# ---------- Has Company ----------

df['has_company'] = df['company'].notnull().astype(int)

# ---------- Room Changed ----------

df['room_changed'] = (
    df['reserved_room_type'] != df['assigned_room_type']
).astype(int)

# ---------- Family Booking ----------

df['is_family'] = (
    (df['children'] > 0) | (df['babies'] > 0)
).astype(int)

# ---------- Lead Time Category ----------

df['lead_time_category'] = pd.cut(
    df['lead_time'],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=[
        '0-30 Days',
        '31-90 Days',
        '91-180 Days',
        '181-365 Days',
        '365+ Days'
    ]
)

# --------------------------------------------
# 6. Final Data Check
# --------------------------------------------

print("\nFinal Dataset Shape:", df.shape)

print("\nMissing Values After Cleaning:\n")
print(df.isnull().sum())

print("\nUpdated Data Types:\n")
print(df.dtypes)

# Preview cleaned data
print("\nCleaned Dataset Preview:\n")
print(df.head())

# --------------------------------------------
# 7. Export Cleaned Dataset
# --------------------------------------------

df.to_csv(
    'data/hotel_bookings_cleaned.csv',
    index=False
)

print("\nCleaned dataset exported successfully.")